In [1]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

In [2]:
retrivermodel= SentenceTransformer('all-MiniLM-L6-v2')

In [3]:
def build_faiss_idx(evidence_corpus):
    embeddings= retrivermodel.encode(evidence_corpus, convert_to_tensor=True)
    index= faiss.IndexFlatIP(embeddings.shape[1])
    index.add(embeddings.cpu().numpy())
    faiss.write_index(index, "evidence_index.faiss")
    return index

In [4]:
def retrieve_evidence(claim, index, evidence_corpus, top_k=10):
    claim_embedding = retrivermodel.encode([claim])
    distances, indices = index.search(claim_embedding, top_k)
    retrieved_docs = [evidence_corpus[i] for i in indices[0]]
    return retrieved_docs

In [6]:
evidence_dataset = ["The sky is blue because of Rayleigh scattering.", "The Eiffel Tower is in Paris, France.", "The moon is made of rock.", ...]

# You would run this once to create your index file
faiss_index = build_faiss_idx(evidence_dataset)

# For every new user claim, you would do this:
# 1. Load your pre-built index
loaded_index = faiss.read_index("evidence_index.faiss")

# 2. Get the user's claim
user_claim = "Why is the sky blue?"

# 3. Retrieve the evidence
potential_evidence = retrieve_evidence(user_claim, loaded_index, evidence_dataset)

# 4. Pass this 'potential_evidence' to your reranker function
print(potential_evidence)

['The sky is blue because of Rayleigh scattering.', 'The moon is made of rock.', Ellipsis, 'The Eiffel Tower is in Paris, France.', Ellipsis, Ellipsis, Ellipsis, Ellipsis, Ellipsis, Ellipsis]
